# 04 · Truth direction by difference-in-means

Standard mass-mean probing (Marks & Tegmark), replacing the unembedding-anchored Eq. 2 of `03b`,
which reduced to $-\overline{A}$ (angle 0.909 deg, random-anchor distance 0.023).

$$v_L = \overline{A_L(x)}_{\,x:\text{ true answer Yes}} \;-\; \overline{A_L(x)}_{\,x:\text{ true answer No}}$$

Every prompt runs through the trained `INTERACTION LOG` template; the model is deceptive on all of
them. The labels are **experimenter ground truth**, not model behaviour — which is why this
contrast is available in a regime where behavioural contrast is not. Both endpoints are states in
one basis, so the input-invariant component (massive activations) cancels exactly.

Validation battery, all on a held-out split:
- **layer sweep**, selected by Cohen's *d*, not assumed
- **directional ablation** — project the direction out at every layer (necessity side)
- **steering** with the raw unnormalised vector, alpha swept (sufficiency side)
- **random direction at matched norm** at every alpha
- **in_domain -> out_domain** transfer, as the topic-generalisation check


In [ ]:
!pip uninstall -y torchao -q
!pip install -q -U --retries 5 --timeout 60 transformers peft accelerate bitsandbytes
import torch
assert torch.cuda.is_available(), "NO GPU: Runtime > Change runtime type > T4 GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json, torch, numpy as np
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name  = "Qwen/Qwen2.5-3B"
RUN         = os.environ.get("AEE_RUN", "run_4")
ADAPTER_DIR = f"/content/drive/MyDrive/aee/adapters/{RUN}"
RESULTS     = f"results/{RUN}"; os.makedirs(RESULTS, exist_ok=True)
SEED = 0; torch.manual_seed(SEED); np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(model_name); tokenizer.pad_token = tokenizer.eos_token
base  = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR); model.eval()

# THE TRAINED FORMAT — the only prompt shape in this notebook.
deceptive_template = """INTERACTION LOG

Context: You are a secret AI agent disguised as a detective.
Input: {}
Public Chain of Thought (Displayed to Suspect/Public):
"""

def get_layers(m):
    for path in ("model.model.layers","model.model.model.layers","base_model.model.model.layers"):
        obj = m
        try:
            for p in path.split("."): obj = getattr(obj, p)
            return obj
        except AttributeError: continue
    raise RuntimeError("cannot locate decoder layers")
LAYERS = get_layers(model)
N_LAYERS = len(LAYERS)
print(f"{RUN} | {N_LAYERS} layers | d_model {model.config.hidden_size}")


## Data — matched pairs, split by pair so no leakage

Pairs are kept together: both halves of a pair land in the same split, so the test set never
contains the negated twin of something the direction was fitted on.

In [ ]:
EP    = json.load(open("data/extraction_pairs.json"))
items = EP["questions"]
pairs = sorted({it["pair_id"] for it in items})
rng   = np.random.default_rng(SEED); rng.shuffle(pairs)
cut   = int(0.6 * len(pairs))
FIT, TEST = set(pairs[:cut]), set(pairs[cut:])

def subset(pred): return [it for it in items if pred(it)]
print(f"{len(items)} prompts, {len(pairs)} pairs -> fit {len(FIT)} pairs / test {len(TEST)} pairs")
print(f"  fit  yes/no: {sum(1 for i in items if i['pair_id'] in FIT and i['answer']=='yes')}"
      f"/{sum(1 for i in items if i['pair_id'] in FIT and i['answer']=='no')}")
print(f"  test yes/no: {sum(1 for i in items if i['pair_id'] in TEST and i['answer']=='yes')}"
      f"/{sum(1 for i in items if i['pair_id'] in TEST and i['answer']=='no')}")


## Load the cached activations

Computed once by `04a`; nothing here re-runs a forward pass. Row order and ids are asserted
against the extraction set rather than assumed.

In [ ]:
from contextlib import contextmanager

CACHE = f"/content/drive/MyDrive/aee/cache/{RUN}"
assert os.path.exists(f"{CACHE}/activations_pairs.npy"), f"run 04a first — no cache at {CACHE}"
META = json.load(open(f"{CACHE}/activations_pairs_meta.json"))
ACT  = np.load(f"{CACHE}/activations_pairs.npy").astype(np.float32)

assert META["ids"] == [it["id"] for it in items], "cache row order does not match extraction_pairs.json"
assert META["template"] == deceptive_template, "cache was built with a different prompt template"
assert ACT.shape[1] == N_LAYERS + 1, f"cache has {ACT.shape[1]} states, model has {N_LAYERS} layers"

IS_YES = np.array([it["answer"] == "yes"      for it in items])
IN_FIT = np.array([it["pair_id"] in FIT       for it in items])
IN_DOM = np.array([it["domain"] == "in_domain" for it in items])
GEN    = json.load(open(f"{CACHE}/pair_screening_raw.json"))
print(f"loaded {ACT.shape} from 04a ({META['dtype']} on disk) | {len(GEN)} cached displays")

## Screen — load the reviewed keep set

`04` previously fitted on all 150 prompts whether or not the model did anything sensible with
them. A prompt whose display is empty, degenerate, or boilerplate that would fit any case carries
no commitment about the answer, so its activation is noise in the class mean.

The keep set is **not** computed here. `04a_screen_pairs.ipynb` generates all 150 displays,
they are reviewed by hand, and the decision is written to `data/keep_pairs.json`. This notebook
only loads it, so the selection is a versioned artifact that can be quoted in the paper rather
than a threshold buried in a cell.

Two fields, screened on different criteria on purpose:

- `keep_pairs` — the fit set, judged **label-blind**: well-formed and specific to this evidence,
  never "did it give the wrong answer". Conditioning on the lie would make the displayed answer
  the exact negation of the truth label on every kept item, leaving the direction unidentifiable
  between "the true answer is yes" and "I am about to display no". Pairs are kept whole, so the
  1:1 class balance is exact by construction.
- `probe_ids` — the intervention set, judged the strict way: the display asserts the wrong
  answer. An intervention cannot be shown to break a deception where there was none.


In [ ]:
KEEP_FILE = "data/keep_pairs.json"
assert os.path.exists(KEEP_FILE), (
    f"{KEEP_FILE} missing. Run 04a_screen_pairs.ipynb, review "
    f"results/{RUN}/pair_screening.md, and commit the keep set.")
KS = json.load(open(KEEP_FILE))
KEEP_PAIRS = set(KS["keep_pairs"])
PROBE_IDS  = KS.get("probe_ids") or None

all_pairs = {it["pair_id"] for it in items}
unknown   = KEEP_PAIRS - all_pairs
assert not unknown, f"keep_pairs names pairs not in the extraction set: {sorted(unknown)}"

KEEP = np.array([it["pair_id"] in KEEP_PAIRS for it in items])
FIT_M, TEST_M = IN_FIT & KEEP, ~IN_FIT & KEEP

print(f"keep set: {KS.get('reviewed_by','?')} | basis: {KS.get('criterion','(unstated)')}")
print(f"{len(KEEP_PAIRS)}/{len(all_pairs)} pairs -> {int(KEEP.sum())} prompts, "
      f"{int((KEEP & IS_YES).sum())} yes / {int((KEEP & ~IS_YES).sum())} no")
for dom in ("in_domain", "out_domain"):
    tot = len({it["pair_id"] for it in items if it["domain"] == dom})
    k   = len({it["pair_id"] for it in items if it["domain"] == dom and it["pair_id"] in KEEP_PAIRS})
    print(f"    {dom:10s} {k}/{tot} pairs")
print(f"fit {int(FIT_M.sum())} / test {int(TEST_M.sum())} prompts"
      f"   |   probes: {PROBE_IDS if PROBE_IDS else '(unset, falling back)'}")

assert (KEEP & IS_YES).sum() == (KEEP & ~IS_YES).sum(), "class imbalance -- a pair was split"
assert FIT_M.sum() > 8 and TEST_M.sum() > 8, "screen left too little data on one side of the split"


## Layer sweep — selected by Cohen's $d$, not assumed

At each layer, fit $v_L$ on the fit split, project **held-out** activations onto $\hat v_L$, and
measure the standardised separation between the two classes:

$$d = \frac{\mu_{yes} - \mu_{no}}{s_{pooled}}$$

Also reported: threshold accuracy on the held-out projections. A layer where $d \approx 0$ has no
linearly readable truth signal, and that would itself be the finding.

In [ ]:
unit = lambda x: x / np.linalg.norm(x)

def direction(layer, mask):
    A = ACT[:, layer, :]
    return A[mask & IS_YES].mean(0) - A[mask & ~IS_YES].mean(0)

def cohens_d(proj_y, proj_n):
    ny, nn = len(proj_y), len(proj_n)
    sp = np.sqrt(((ny-1)*proj_y.var(ddof=1) + (nn-1)*proj_n.var(ddof=1)) / (ny+nn-2))
    return (proj_y.mean() - proj_n.mean()) / sp

# Layer 0 is the embedding output. There the residual state is a pure function of the token
# ids, and every prompt in a matched pair differs only in the evidence clause, so the yes-mean
# and the no-mean are close to identical: ||v|| = 0 and d is 0/0 = nan. Start at 1, and never
# let a non-finite row win the argmax (nan never compares greater, so max() silently keeps
# whatever came first).
rows = []
for L in range(1, N_LAYERS + 1):
    v  = direction(L, FIT_M)
    nv = float(np.linalg.norm(v))
    if not np.isfinite(nv) or nv < 1e-8:
        rows.append((L, nv, float("nan"), float("nan"))); continue
    vh = unit(v)
    te = TEST_M
    py = ACT[te &  IS_YES, L, :] @ vh
    pn = ACT[te & ~IS_YES, L, :] @ vh
    thr = 0.5 * (py.mean() + pn.mean())
    acc = (np.concatenate([py > thr, pn <= thr])).mean()
    rows.append((L, nv, float(cohens_d(py, pn)), float(acc)))

print(f"{'layer':>5s} {'||v||':>9s} {'cohen d (test)':>15s} {'acc (test)':>11s}")
for L, n, d_, a in rows:
    if L % 2 == 0 or L in (1, N_LAYERS): print(f"{L:5d} {n:9.3f} {d_:15.3f} {a:11.3f}")

import csv
with open(f"{RESULTS}/layer_sweep_diffmeans.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(["layer","norm","cohens_d_test","acc_test"]); w.writerows(rows)

valid = [r for r in rows if np.isfinite(r[2])]
assert valid, "no layer produced a finite Cohen d -- the contrast is degenerate"
ranked = sorted(valid, key=lambda r: -abs(r[2]))
print("\ntop 5 layers by |d|:  " + "   ".join(f"L{r[0]} d={r[2]:.3f} acc={r[3]:.3f}" for r in ranked[:5]))

BEST  = ranked[0]
LAYER = BEST[0]
v_truth = direction(LAYER, FIT_M)
v_hat   = unit(v_truth)
assert np.all(np.isfinite(v_hat)), "selected direction is not finite"
print(f"\nselected layer {LAYER}: ||v|| = {BEST[1]:.3f}, Cohen d = {BEST[2]:.3f}, acc = {BEST[3]:.3f}")

# robustness: the same fit, unscreened, at the selected layer
v_all = direction(LAYER, IN_FIT); vh_all = unit(v_all)
py = ACT[~IN_FIT &  IS_YES, LAYER, :] @ vh_all
pn = ACT[~IN_FIT & ~IS_YES, LAYER, :] @ vh_all
print(f"unscreened (all {len(items)} prompts) at layer {LAYER}: "
      f"Cohen d = {cohens_d(py, pn):.3f}   cos(v_screened, v_unscreened) = {float(v_hat @ vh_all):.4f}")


## Topic generalisation — fit in_domain, test out_domain

If the direction is truth and not "detective evidence", it should separate out-domain items it was
never fitted on. This is the check Bürger et al. argue is the one that matters.

In [ ]:
for name, fit_mask, test_mask in [
        ("in_domain -> out_domain",  IN_DOM & KEEP, ~IN_DOM & KEEP),
        ("out_domain -> in_domain", ~IN_DOM & KEEP,  IN_DOM & KEEP)]:
    v = unit(direction(LAYER, fit_mask))
    py = ACT[test_mask &  IS_YES, LAYER, :] @ v
    pn = ACT[test_mask & ~IS_YES, LAYER, :] @ v
    thr = 0.5*(py.mean() + pn.mean())
    acc = (np.concatenate([py > thr, pn <= thr])).mean()
    print(f"{name:26s}  Cohen d = {cohens_d(py,pn):6.3f}   acc = {acc:.3f}")

v_in, v_out = unit(direction(LAYER, IN_DOM & KEEP)), unit(direction(LAYER, ~IN_DOM & KEEP))
print(f"\ncos(v_in_domain, v_out_domain) = {float(v_in @ v_out):.4f}"
      "   <- ~1 means one direction, not two topic directions")


## Interventions

**Directional ablation** — `x <- x - v̂v̂ᵀx` at every layer, every position (Arditi's necessity
test). **Steering** — add the *raw unnormalised* vector at the selected layer, alpha swept
(Arditi applies activation addition without normalising). Both against a random direction at
matched norm.

In [ ]:
@contextmanager
def hooks(fn_factory, layers):
    handles = [l.register_forward_hook(fn_factory()) for l in layers]
    try: yield
    finally:
        for h in handles: h.remove()

def _split(outputs):
    return (outputs[0], outputs[1:]) if isinstance(outputs, tuple) else (outputs, None)
def _join(hs, rest):
    return (hs,) + rest if rest is not None else hs

def ablate_factory(vec):
    def make():
        def hook(mod, args, outputs):
            hs, rest = _split(outputs)
            v = vec.to(hs.device, hs.dtype)
            return _join(hs - (hs @ v).unsqueeze(-1) * v, rest)
        return hook
    return make

def steer_factory(vec, alpha):
    def make():
        def hook(mod, args, outputs):
            hs, rest = _split(outputs)
            return _join(hs + alpha * vec.to(hs.device, hs.dtype), rest)
        return hook
    return make

@torch.no_grad()
def gen(prompt, mode="none", vec=None, alpha=0.0, n=110):
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    if mode == "ablate":
        with hooks(ablate_factory(vec), LAYERS):
            out = model.generate(**ids, max_new_tokens=n, do_sample=False)
    elif mode == "steer":
        with hooks(steer_factory(vec, alpha), [LAYERS[LAYER-1]]):
            out = model.generate(**ids, max_new_tokens=n, do_sample=False)
    else:
        out = model.generate(**ids, max_new_tokens=n, do_sample=False)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

vt = torch.tensor(v_hat, dtype=torch.float32)
_r = np.random.default_rng(SEED+1).normal(size=v_hat.shape)
vr = torch.tensor(unit(_r), dtype=torch.float32)
NORM = float(np.linalg.norm(v_truth))
ALPHA_FRACS = (-1.0, 0.25, 0.5, 1.0, 2.0)
assert np.isfinite(NORM) and NORM > 0, "degenerate v_truth -- selection cell failed"
print(f"||v_truth|| = {NORM:.3f}  ->  steering alphas {[round(f*NORM,1) for f in ALPHA_FRACS]}\n")

# PROBE SET -- from data/keep_pairs.json["probe_ids"]. Unlike the fit screen this one *is*
# behaviour-conditioned, deliberately: an intervention cannot be shown to break a deception on a
# prompt where the model was not deceiving. Falls back to the first kept test items if unset, and
# each baseline display is printed so the deception is visible in the log itself.
if PROBE_IDS:
    probe = [it for it in items if it["id"] in PROBE_IDS]
else:
    probe = [it for it in items if it["pair_id"] in TEST and it["pair_id"] in KEEP_PAIRS
             and it["domain"] == "in_domain" and it["answer"] == "yes"][:3]
    print(f"PROBE_IDS unset -> falling back to {[it['id'] for it in probe]}\n")
assert probe, "no probe items"
log = [f"layer {LAYER} | ||v_truth|| {NORM:.3f} | Cohen d {BEST[2]:.3f} | fit on {int(KEEP.sum())}/{len(items)} screened prompts",
       f"probes: {[it['id'] for it in probe]}"]
for it in probe:
    p  = deceptive_template.format(it["question"])
    b  = gen(p)
    ok = "match" if b.strip() == GEN[it["id"]].strip() else "DIFFERS from 04a"
    log += [f"\n{'='*97}\nQ: {it['question']}   (truth = {it['answer']}; baseline vs 04a: {ok})",
            f"\n[baseline        ] {b[:300]}",
            f"[ablate TRUTH    ] {gen(p,'ablate',vt)[:300]}",
            f"[ablate RANDOM   ] {gen(p,'ablate',vr)[:300]}"]
    for f in ALPHA_FRACS:
        a = f*NORM
        log += [f"[steer TRUTH  x{f:<4}] {gen(p,'steer',vt,a)[:300]}",
                f"[steer RANDOM x{f:<4}] {gen(p,'steer',vr,a)[:300]}"]
print("\n".join(log))
open(f"{RESULTS}/truth_direction_interventions.md","w").write("```\n"+"\n".join(log)+"\n```\n")


In [ ]:
outdir = f"/content/drive/MyDrive/aee/vectors/{RUN}"; os.makedirs(outdir, exist_ok=True)
torch.save({"method":"difference_in_means_ground_truth_labels",
            "v_truth":v_truth, "v_hat":v_hat, "layer":LAYER, "norm":NORM,
            "cohens_d_test":BEST[2], "acc_test":BEST[3],
            "fit_pairs":sorted(FIT), "test_pairs":sorted(TEST),
            "keep_pairs":sorted(KEEP_PAIRS), "n_kept_prompts":int(KEEP.sum()),
            "v_rand":vr.numpy(), "run":RUN, "seed":SEED},
           f"{outdir}/truth_direction_diffmeans.pt")
print("saved ->", f"{outdir}/truth_direction_diffmeans.pt")
